# Modulo 4 — Strategy Generator

Trasforma i pattern stabili del Modulo 3 in **strategie complete**: entry (appartenenza al pattern), exit gestita barra per barra con **Stop Loss, Take Profit, Break-Even, Trailing Stop**, **time-stop**, dimensionamento a rischio fisso e **filtri** (trend, volatilità, orari, max operazioni).

Tutte le distanze SL/TP sono in **multipli di ATR** → la strategia si adatta da sola alla volatilità dello strumento. Il backtester è **conservativo** (in caso di ambiguità intrabar assume lo stop prima del target).

In [ ]:
import sys
from pathlib import Path
for c in [Path.cwd(), Path.cwd().parent, Path('/kaggle/working/trading-ai')]:
    if (c / 'trading_ai').exists():
        sys.path.insert(0, str(c)); break

from trading_ai.data_engine import DataEngine, generate_ohlcv
from trading_ai.feature_engineering import FeatureEngine
from trading_ai.pattern_discovery import PatternDiscovery
from trading_ai.strategy_generator import StrategyGenerator, RiskParams, Filters

## 1. Pipeline Moduli 1-2-3

In [ ]:
eng = DataEngine()
h1 = eng.to_timeframe(eng.load_dataframe(generate_ohlcv(n=200_000, seed=3)), 'H1')
feats = FeatureEngine().transform(h1, dropna=True)
disc = PatternDiscovery(n_clusters=25, horizon=10, min_profit_factor=1.05, min_count_test=15)
disc.discover(feats)
print('Pattern stabili:', len(disc.stable_patterns()))

## 2. Costruzione strategie + risk management

Definisci una politica di rischio e i filtri; il generatore crea una strategia per ogni pattern stabile.

In [ ]:
risk = RiskParams(sl_atr=2.0, tp_atr=3.0, be_atr=1.0, trail_atr=1.5,
                  risk_per_trade=0.01, max_bars=40, max_trades_per_day=3)
filters = Filters(use_trend=True, min_adx=15.0, start_hour=7, end_hour=20)

gen = StrategyGenerator(disc, risk=risk, filters=filters)
strategies = gen.build()
for s in strategies:
    print(s.name, '->', s.describe()['risk'])

## 3. Backtest di tutte le strategie

Metriche aggregate per strategia (anticipo del Modulo 9): n. trade, win rate, profit factor, expectancy, rendimento totale, max drawdown.

In [ ]:
summary = gen.backtest_all(feats)
summary

## 4. Dettaglio di una strategia (curva equity)

In [ ]:
if strategies:
    res = strategies[0].run(feats)
    print('Trade:', len(res.trades))
    res.equity.plot(title=f'Equity - {strategies[0].name}', figsize=(10, 4))

## ✅ Modulo 4 verificato

Strategie complete con gestione del rischio e filtri, backtester event-driven verificato con scenari deterministici (`pytest tests/test_strategy_generator.py`, 7/7).

> Su dati sintetici i rendimenti sono casuali: l'obiettivo qui è la **correttezza della meccanica**, non la profittabilità. Su dati reali userai il Modulo 5 per validare la robustezza.

**Prossimo:** Modulo 5 — Validation (Walk-Forward, Out-of-Sample, Monte Carlo, Robustness, Sensitivity) per scartare le strategie non robuste.